<a href="https://colab.research.google.com/github/NidhiCN24/Fake-News-Detection/blob/main/Fake_news_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [32]:
#Fake news classification model
#Installation
!pip -q install transformers datasets scikit-learn gradio

import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import gradio as gr

In [33]:
import os
import csv
import sys

In [36]:
csv.field_size_limit(sys.maxsize)

df = pd.read_csv(
    '/content/WELFake_Dataset.csv',
    engine='python',
    encoding='utf-8',
    sep=',',
    quotechar='"',
    escapechar='\\',
    on_bad_lines='warn'
)

# Basic cleaning
df = df[['text','label']].dropna()
#Removing extremely short rows as they do not contribute much to the model understanding and leads to increased training time
df = df[df['text'].str.len() > 20]
df = df.sample(15000, random_state=42)
#converting the label to integers (as earlier I got float value as the label)
df['label'] = df['label'].astype(int)

print(df.head())
print("\nLabel Distribution:\n", df['label'].value_counts())

/tmp/ipykernel_1643/3876984807.py:3: ParserWarning: Skipping line 8580: ',' expected after '"'

  df = pd.read_csv(
/tmp/ipykernel_1643/3876984807.py:3: ParserWarning: Skipping line 8580: Expected 4 fields in line 8580, saw 8

  df = pd.read_csv(


                                                    text  label
22599  President Trump addressed a joint session of C...      0
13920  MSNBC reporter Kasie Hunt began her segment on...      1
63401  You know things are going badly for the GOP Pr...      1
33354  Barack Obama, Eric Holder, Al Sharpton, Hillar...      1
47555  I dare you to restrain yourself from laughing ...      1

Label Distribution:
 label
1    7535
0    7465
Name: count, dtype: int64


In [37]:
#train test split
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)

In [38]:
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

def tokenize(batch):
    return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=256)

train_ds = Dataset.from_pandas(train_df).map(tokenize, batched=True)
val_ds = Dataset.from_pandas(val_df).map(tokenize, batched=True)

train_ds.set_format(type='torch', columns=['input_ids','attention_mask','label'])
val_ds.set_format(type='torch', columns=['input_ids','attention_mask','label'])

Map:   0%|          | 0/12000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

In [50]:
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

model.config.id2label = {0: "Real News", 1: "Fake News"}
model.config.label2id = {"Real News": 0, "Fake News": 1}

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [40]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": (preds == labels).mean()
    }

In [41]:
#training the model
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    logging_steps=100,
    save_strategy="no"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.054501,0.045221,0.984667
2,0.029020,0.065654,0.983333
3,0.009207,0.079030,0.984333


TrainOutput(global_step=2250, training_loss=0.04786417775476972, metrics={'train_runtime': 915.226, 'train_samples_per_second': 39.335, 'train_steps_per_second': 2.458, 'total_flos': 2384413175808000.0, 'train_loss': 0.04786417775476972, 'epoch': 3.0})

In [42]:
#model evaluation
preds = trainer.predict(val_ds)
y_pred = np.argmax(preds.predictions, axis=1)
y_true = preds.label_ids

print(classification_report(y_true, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.98      0.99      0.98      1493
           1       0.99      0.98      0.98      1507

    accuracy                           0.98      3000
   macro avg       0.98      0.98      0.98      3000
weighted avg       0.98      0.98      0.98      3000

Confusion Matrix:
 [[1479   14]
 [  33 1474]]


In [43]:
#samples that are misclassified
val_df = val_df.reset_index(drop=True)
wrong = np.where(y_pred != y_true)[0][:5]

print("\nMisclassified Samples:\n")
for i in wrong:
    print("Text:", val_df['text'][i][:200])
    print("Actual:", y_true[i], "Pred:", y_pred[i])
    print()


Misclassified Samples:

Text: 
Win or lose, Trump has left his mark, and America has changed.
On the eve of the election, many Republican insiders and loyal commentators are acknowledging what has become apparent – that rise of Tr
Actual: 1 Pred: 0

Text: EXCLUSIVE: A senior Hillary Clinton aide has maintained her top secret security clearance despite sending information now deemed classified to the Clinton Foundation and to then-Secretary of State Cli
Actual: 0 Pred: 1

Text: 
On Monday Bill Clinton campaigned in Greensboro, North Carolina for his wife Hillary Clinton.
Bill was interrupted by several Black Lives Matter protesters chanting “black lives matter,” while holdin
Actual: 1 Pred: 0

Text: Tesla Motors is readying improvements to its Autopilot technology that might have prevented an accident in May that took the life of an Ohio man, Tesla’s chief executive said on Sunday. The man was ki
Actual: 0 Pred: 1

Text: Existing Home Sales- Another Reason Why You Should Take the Money

In [44]:
#saving the model
model.save_pretrained('/content/fake_news_model')
tokenizer.save_pretrained('/content/fake_news_model')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/fake_news_model/tokenizer_config.json',
 '/content/fake_news_model/tokenizer.json')

In [45]:
#improvement - weighted loss
class_counts = df['label'].value_counts()
weights = torch.tensor([1/class_counts[0], 1/class_counts[1]], dtype=torch.float)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fn = torch.nn.CrossEntropyLoss(weight=weights.to(model.device))
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

trainer_w = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds
)
trainer_w.train()

Epoch,Training Loss,Validation Loss
1,0.019962,0.096528
2,0.002048,0.096576
3,0.003673,0.112435


TrainOutput(global_step=2250, training_loss=0.012680830713141605, metrics={'train_runtime': 917.295, 'train_samples_per_second': 39.246, 'train_steps_per_second': 2.453, 'total_flos': 2384413175808000.0, 'train_loss': 0.012680830713141605, 'epoch': 3.0})

In [47]:
print(model.config.id2label)

{0: 'Fake News', 1: 'Real News'}


In [51]:
import torch

def predict(text):
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    )

    # Move to same device as model
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        pred = torch.argmax(outputs.logits, dim=1).item()
    if(pred == 1):
      final_prediction = "Real News"
    else:
      final_prediction = "Fake News"

    return final_prediction

import gradio as gr

iface = gr.Interface(
    fn=predict,
    inputs=gr.Textbox(lines=3, placeholder="Enter news text here..."),
    outputs="text"
)

iface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://57993294ed001db9c7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
